# Plot Shivdasani

In [ ]:
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import marsilea as ma
import marsilea.plotter as mp
from pathlib import Path

# Global Scanpy settings
sc.settings.verbosity = 2               # Show progress
sc.settings.set_figure_params(
    dpi=150,                            # High-res output
    dpi_save=300,                       # High-res when saving
    format='pdf',                       # or 'pdf', 'svg'
    facecolor='white',                  # or 'none' for transparent
    frameon=False,                      # No outer frame
    vector_friendly=True,               # No rasterization warnings
    fontsize=10,                        # Base font size
    figsize=(4, 4),                     # Default figure size
    transparent=True                    # Transparent background if saving
)

mpl.rcParams.update({
    "svg.fonttype": "none",       
    "pdf.fonttype": 42,           
    "legend.fontsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})

sc.settings.autoshow = False
sc.settings.figdir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/External/Shivdasani")

# Set base directory for data
base_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/external/shivdasani")

In [ ]:
cell_type_colors = {
    # Lightest to darkest (grouped & distinct, no very dark shades)

    # Ambiguous/general
    'AS + ND1':         '#ff7f00',  # Bright orange

    # Transcription factor-expressing (blue–cyan)
    'NEUROD1':    '#00cfc1',  # Bright turquoise
    'IRF':        '#1fa2a5',  # Teal
    'ASCL1':      '#2171b5',  # Medium blue
    'HES6':       '#3182bd',  # Clear sky blue

    # Stem/progenitor-like (greens)
    'CREB3L1':    '#a6d854',  # Lime green
    'ISC':        '#33a02c',  # Forest green

    # Neuroendocrine lineage (magentas & purples)
    'EC-cells':   '#f768a1',  # Light pink
    'T4-cells':   '#e7298a',  # Medium pink
    'D-cells':    '#ce1256',  # Vivid rose
    'X-cells':    '#980043',  # Strong magenta (no longer near-black)
}

## Load data

In [ ]:
adata = sc.read(base_dir / "seurat.processed.annotated.h5ad")
adata.raw = adata.copy()

In [ ]:
adata.var.loc["FABP1"]

In [ ]:
adata.obs["celltype"] = adata.obs["paper.annotation"].astype("category")
adata.obs["celltype"] = adata.obs["celltype"].cat.reorder_categories(list(cell_type_colors.keys()), ordered=True)
adata.uns["celltype_colors"] = [cell_type_colors[ct] for ct in list(cell_type_colors.keys())]

In [ ]:
adata.X.max()

In [ ]:
# Identify mitochondrial and ribosomal genes by name
gene_names = adata.var_names.str.upper()
is_mito = gene_names.str.startswith("MT-")  # Human
is_ribo = gene_names.str.startswith(("RPS", "RPL"))  # Ribosomal proteins

adata = adata[:, ~is_mito & ~is_ribo].copy()  # Exclude mito and ribo genes

# Identify Highly Variable Genes (HVG)
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    n_top_genes=3000
)

# Keep only HVGs
adata = adata[:, adata.var["highly_variable"]].copy()

# Scale data (zero mean, unit variance)
sc.pp.scale(adata, max_value=10)

In [ ]:
# PCA
sc.tl.pca(adata)
# Neighbor calculation
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
# UMAP
sc.tl.umap(adata)

In [ ]:
# Perform harmony integration
sc.external.pp.harmony_integrate(adata, "orig.ident")

In [ ]:
# Neighbor calculation
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
# UMAP
sc.tl.umap(adata, key_added="X_umap_harmony")

In [ ]:
sc.pl.embedding(adata, color="celltype", palette=cell_type_colors, basis="X_umap_harmony", frameon=False, save="_paper_annotation.pdf", show=True)

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
from scipy import sparse

features = ["NEUROG3", "GCG", "CCK", "PYY", 
            "GHRL", "MLN", "SST", "GIP", 
            "TPH1", "LGR5", "MUC2", "FABP1"]

plot_features = []

for gene in features:
    x = adata[:, gene].X

    if sparse.issparse(x):
        x = x.toarray()

    x = np.asarray(x).flatten().astype(float)

    # set zero / missing expression to NaN
    x[x == 0] = np.nan

    col = f"{gene}_plot"
    adata.obs[col] = x
    plot_features.append(col)

sc.pl.embedding(
    adata,
    color=plot_features,
    cmap="Reds",
    na_color="#cccccc",   # grey for zero/NaN cells
    basis="X_umap_harmony",
    ncols=4,
    frameon=False,
    save="_features.pdf",
    show=True
)

## Load stomach tissue atlas

In [ ]:
# Load stomach atlas
adata_stomach = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/stomach_standalone/processed/scanvi_stomach_adult_fetal_ref.h5ad")
adata_stomach = adata_stomach.raw.to_adata()

In [ ]:
# Subset to adult 
adata_stomach = adata_stomach[adata_stomach.obs["dataset"] == "adult"].copy()

In [ ]:
adata_stomach.raw = adata_stomach.copy()
adata_stomach.layers["counts"] = adata_stomach.X.copy()

sc.pp.normalize_total(adata_stomach, target_sum=1e4)
sc.pp.log1p(adata_stomach)
adata_stomach.layers["lognorm"] = adata_stomach.X.copy()

In [ ]:
# Subset to X cells
adata_stomach = adata_stomach[adata_stomach.obs["cell_type"] == "X cells"].copy()

In [ ]:
with plt.rc_context({"axes.grid": False}):
    fig = sc.pl.violin(
        adata_stomach,
        keys=["GHRL", "MLN"],
        groupby="cell_type",
        layer="lognorm",
        palette={"X cells": "#980043"},
        show=False,
        use_raw=False
    )

    plt.gcf().set_size_inches(6, 4)
    plt.savefig(sc.settings.figdir / "stomach_x_cells.pdf", bbox_inches="tight")
    plt.show()

## Load intestine tissue atlas

In [ ]:
# Load the space time gut cell atlas (epithelial subset)
adata_intestine = sc.read( "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/external/gut_cell_atlas/epi_log_counts02_v2.h5ad")

In [ ]:
list(adata_intestine.obs["annotation"].unique())

In [ ]:
# Subset to M/X cells (MLN/GHRL+)
adata_intestine = adata_intestine[adata_intestine.obs["annotation"].isin(["M/X cells (MLN/GHRL+)"])].copy()

In [ ]:
with plt.rc_context({"axes.grid": False}):
    fig = sc.pl.violin(
        adata_intestine,
        keys=["GHRL", "MLN"],
        groupby="annotation",
        palette={"M/X cells (MLN/GHRL+)": "#980043"},
        show=False,
        use_raw=False
    )

    plt.gcf().set_size_inches(6, 4)
    plt.savefig(sc.settings.figdir / "intestine_x_cells.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
sc.settings.figdir